# Datamine PERTRA Process Example

This notebook demonstrates how to configure and run the **`pertra`** process wrapper in `dmstudio`.

## Process Description

## PERTRA

Generate perimeters in planes across other perimeters.

PERTRA processes a file of input perimeters, and generates the corresponding perimeters for a set of parallel planes which intersect the planes of the input perimeters.

* Input perimeters need not be convex but must not intersect themselves.

* There can be more than one perimeter for each input plane.

* Input perimeters in the same plane can enclose but not intersect each other.

* Input perimeter planes need not be parallel to each other or perpendicular to the output planes.

* Each input perimeter can carry an ore-body identification number (TAG) which will ensure that all points on an output perimeter refer to the same ore body.

### Input Files:

* **perimin** (String):
  The input perimeter file. This file must contain the fields declared in the command
  line as **PLANE, PVALUE, PTN, X, Y** and **Z**. Note that X, Y and Z are GLOBAL
  coordinates. If **TAG** is declared in the command line, it must appear in this file.
  This file must be sorted into **PLANE, PVALUE, PTN** order.
  Required=Yes

* **intersec** (Undefined):
  An optional output file of the lines of points produced where the input perimeters
  intersect the output perimeter planes. The output perimeters are constructed by
  arranging these points into lists. This file will contain fields **PLANE, LN, X, Y** and
  **Z**. Note that X, Y and Z are GLOBAL coordinates. A tag field will only appear if
  **TAG** is declared in the command line.
  Required=No

### Output Files:

* **perimout** (String):
  An optional output file of perimeters in the planes indicated by the parameter values
  below. This file will contain fields **PLANE, PVALUE, PTN, XP, YP** and **ZP**. Note
  that **XP, YP** and **ZP** are GLOBAL coordinates. A tag field will only appear if
  **TAG** is declared in the command line. AT LEAST ONE OUTPUT FILE MUST BE SPECIFIED.
  Required=No

### Fields:

* **plane** (Any : PERIMIN):
  Required field in **PERIMIN**. Plane identifier. This identifier must be such that,
  when the values are in ascending order, the planes are in sequence. **PLANE** will often
  be the same as X, Y or Z but cannot be the same field. It may be necessary to copy a
  field by using **GENTRA**.
  Default=Undefined
  Required=Yes

* **pvalue** (Any : PERIMIN):
  Required field in **PERIMIN**. Perimeter ID value. Note that there can be more than one
  perimeter in a plane but that, on input, they must not cross either themselves or each
  other.
  Default=Undefined
  Required=Yes

* **ptn** (Numeric : PERIMIN):
  Required field in **PERIMIN**. Point number in perimeter.
  Default=Undefined
  Required=Yes

* **x** (Numeric : PERIMIN):
  Required field in **PERIMIN**. Global X-coordinate.
  Default=Undefined
  Required=Yes

* **y** (Numeric : PERIMIN):
  Required field in **PERIMIN**. Global Y-coordinate.
  Default=Undefined
  Required=Yes

* **z** (Numeric : PERIMIN):
  Required field in **PERIMIN**. Global Z-coordinate.
  Default=Undefined
  Required=Yes

* **tag** (Any : PERIMIN):
  Optional field. Ore body identifying value. If declared in the command line, it must
  appear in **PERIMIN** and will appear in the output file(s).
  Default=Undefined
  Required=No

### Parameters:

* **direct**:
  Required parameter which specifies the plane of the output perimeters. 1 = XY, 2 = XZ,

## 3 = YZ

  Range=1,3
  Values=1,2,3
  Default=1
  Required=Yes

* **startpos**:
  Required parameter which specifies the value of the coordinate perpendicular to the
  output plane for the first output plane.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **stize**:
  Required parameter which specifies the distance between output planes.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **numplane**:
  Required parameter which specifies the number of output planes. Note that no harm is
  done if output planes are requested which do not intersect the input perimeters.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **maxdist**:
  Optional parameter which specifies a distance between input planes beyond which ore
  bodies are not to be linked. That is, if two adjacent input planes are more than this
  distance apart where they intersect an output plane, the perimeters on either side of
  the gap will be closed off. The distance between two adjacent input planes is defined
  for this purpose as the distance between the closest pair of points which could
  logically be joined.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('pertra')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute pertra
print("Running pertra...")
dm_cmd.pertra(
    perimin_i='t_input_file',  # required
    plane_f='optional',  # required
    pvalue_f='optional',  # required
    ptn_f='optional',  # required
    x_f='X',  # required
    y_f='Y',  # required
    z_f='Z',  # required
    tag_f='optional',  # required
    direct_p='required_val',  # required
    startpos_p='required_val',  # required
    stize_p='required_val',  # required
    numplane_p='required_val',  # required
    # intersec_i='optional',  # optional
    # perimout_o='t_pertra_out',  # optional
    # maxdist_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("pertra execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_pertra_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")